# Rețele neurale recurente

În modulul anterior, am acoperit reprezentări semantice bogate ale textului. Arhitectura pe care am folosit-o capturează înțelesul agregat al cuvintelor dintr-o propoziție, dar nu ține cont de **ordinea** cuvintelor, deoarece operația de agregare care urmează embedding-urilor elimină această informație din textul original. Deoarece aceste modele nu pot reprezenta ordonarea cuvintelor, ele nu pot rezolva sarcini mai complexe sau ambigue, precum generarea de text sau răspunsul la întrebări.

Pentru a captura înțelesul unei secvențe de text, vom folosi o arhitectură de rețea neurală numită **rețea neurală recurentă**, sau RNN. Când folosim un RNN, trecem propoziția noastră prin rețea câte un token pe rând, iar rețeaua produce un anumit **stare**, pe care o vom transmite din nou rețelei împreună cu token-ul următor.

![Imagine care arată un exemplu de generare cu rețea neurală recurentă.](../../../../../translated_images/ro/rnn.27f5c29c53d727b5.webp)

Având secvența de intrare de tokeni $X_0,\dots,X_n$, RNN creează o secvență de blocuri de rețea neurală și antrenează această secvență end-to-end folosind backpropagation. Fiecare bloc de rețea primește ca intrare o pereche $(X_i,S_i)$ și produce ca rezultat $S_{i+1}$. Starea finală $S_n$ sau ieșirea $Y_n$ intră într-un clasificator liniar pentru a produce rezultatul. Toate blocurile de rețea împart aceleași greutăți și sunt antrenate end-to-end folosind un singur pas de backpropagation.

> Figura de mai sus arată rețeaua neurală recurentă în forma desfăcută (în stânga) și în reprezentarea recurentă mai compactă (în dreapta). Este important să realizăm că toate celulele RNN au aceleași **greutăți partajabile**.

Deoarece vectorii de stare $S_0,\dots,S_n$ sunt transmiși prin rețea, RNN poate învăța dependențe secvențiale între cuvinte. De exemplu, atunci când cuvântul *not* apare undeva în secvență, poate învăța să nege anumite elemente din vectorul de stare.

În interior, fiecare celulă RNN conține două matrici de greutăți: $W_H$ și $W_I$, și un bias $b$. La fiecare pas RNN, dat fiind inputul $X_i$ și starea de intrare $S_i$, starea de ieșire este calculată ca $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, unde $f$ este o funcție de activare (adesea $\tanh$).

> Pentru probleme precum generarea de text (pe care o vom acoperi în unitatea următoare) sau traducerea automată, dorim să obținem și o valoare de ieșire la fiecare pas RNN. În acest caz, există și o altă matrice $W_O$, iar ieșirea se calculează ca $Y_i=f(W_O\times S_i+b_O)$.

Hai să vedem cum rețelele neurale recurente ne pot ajuta să clasificăm setul nostru de date de știri.

> Pentru mediul sandbox, trebuie să rulăm următoarea celulă pentru a ne asigura că biblioteca necesară este instalată și datele sunt preluate în avans. Dacă rulezi local, poți sări peste această celulă.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

La antrenarea modelelor mari, alocarea memoriei GPU poate deveni o problemă. De asemenea, poate fi nevoie să experimentăm cu diferite dimensiuni ale minibatch-ului, astfel încât datele să încapă în memoria GPU, dar antrenamentul să fie suficient de rapid. Dacă rulați acest cod pe propria dvs. mașină cu GPU, puteți experimenta ajustând dimensiunea minibatch-ului pentru a accelera antrenamentul.

> **Notă**: Anumite versiuni ale driverelor NVidia sunt cunoscute pentru faptul că nu eliberează memoria după antrenarea modelului. Rulăm mai multe exemple în acest notebook, iar acest lucru poate cauza epuizarea memoriei în anumite configurații, mai ales dacă faceți propriile experimente în cadrul aceluiași notebook. Dacă întâmpinați erori ciudate la începutul antrenării modelului, ar putea fi util să reporniți kernel-ul notebook-ului.


In [3]:
batch_size = 16
embed_size = 64

## Clasificator RNN simplu

În cazul unui RNN simplu, fiecare unitate recurentă este o rețea liniară simplă, care primește un vector de intrare și un vector de stare, și produce un nou vector de stare. În Keras, acest lucru poate fi reprezentat prin stratul `SimpleRNN`.

Deși putem transmite tokeni codificați one-hot direct către stratul RNN, acest lucru nu este o idee bună din cauza dimensionalității lor ridicate. Prin urmare, vom folosi un strat de embedding pentru a reduce dimensionalitatea vectorilor de cuvinte, urmat de un strat RNN și, în final, un clasificator `Dense`.

> **Notă**: În cazurile în care dimensionalitatea nu este atât de mare, de exemplu atunci când se folosește tokenizarea la nivel de caracter, ar putea avea sens să se transmită tokenii codificați one-hot direct în celula RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Notă:** Folosim aici un strat de embedding neantrenat pentru simplitate, dar pentru rezultate mai bune putem folosi un strat de embedding preantrenat folosind Word2Vec, așa cum este descris în unitatea precedentă. Ar fi un exercițiu bun pentru tine să adaptezi acest cod pentru a funcționa cu embeddings preantrenați.

Acum să antrenăm RNN-ul nostru. În general, RNN-urile sunt destul de dificile de antrenat, deoarece odată ce celulele RNN sunt desfășurate pe lungimea secvenței, numărul rezultat de straturi implicate în backpropagation este destul de mare. Astfel, trebuie să selectăm o rată de învățare mai mică și să antrenăm rețeaua pe un set de date mai mare pentru a obține rezultate bune. Acest lucru poate dura destul de mult timp, deci utilizarea unui GPU este preferată.

Pentru a accelera procesul, vom antrena modelul RNN doar pe titlurile știrilor, omitând descrierea. Poți încerca să antrenezi cu descriere și să vezi dacă poți să antrenezi modelul.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Notă** că acuratețea este probabil mai scăzută aici, deoarece antrenăm doar pe titluri de știri.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Acum că folosim mascare, putem antrena modelul pe întregul set de date de titluri și descrieri.

> **Notă**: Ai observat că am folosit un vectorizator antrenat pe titlurile știrilor, și nu pe întregul corp al articolului? În mod potențial, acest lucru poate face ca unele dintre tokenuri să fie ignorate, așa că este mai bine să reantrenăm vectorizatorul. Totuși, ar putea avea doar un efect foarte mic, așa că vom rămâne la vectorizatorul pre-antrenat anterior de dragul simplității.


## LSTM: Memorie pe termen lung și scurt

Una dintre principalele probleme ale RNN-urilor sunt **gradientele care dispar**. RNN-urile pot fi destul de lungi și pot întâmpina dificultăți în propagarea gradientelor până la primul strat al rețelei în timpul backpropagation-ului. Când se întâmplă acest lucru, rețeaua nu poate învăța relații între tokeni distanțați. O modalitate de a evita această problemă este să introducem **gestionarea explicită a stării** folosind **porți**. Cele două arhitecturi cele mai comune care introduc porți sunt **memoria pe termen lung și scurt** (LSTM) și **unitatea de releu cu poartă** (GRU). Vom acoperi aici LSTM-urile.

![Imagine care arată un exemplu de celulă de memorie pe termen lung și scurt](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

O rețea LSTM este organizată într-un mod similar unui RNN, dar există două stări care sunt transmise de la strat la strat: starea efectivă $c$ și vectorul ascuns $h$. La fiecare unitate, vectorul ascuns $h_{t-1}$ este combinat cu intrarea $x_t$, iar împreună controlează ceea ce se întâmplă cu starea $c_t$ și ieșirea $h_{t}$ prin **porți**. Fiecare poartă are o activare sigmoidă (ieșire în intervalul $[0,1]$), care poate fi considerată o mască bit cu bit atunci când este înmulțită cu vectorul de stare. LSTM-urile au următoarele porți (de la stânga la dreapta în imaginea de mai sus):
* **poarta de uitare** care determină ce componente ale vectorului $c_{t-1}$ trebuie uitate și care să fie păstrate.
* **poarta de intrare** care determină câtă informație din vectorul de intrare și din vectorul ascuns anterior ar trebui încorporată în vectorul de stare.
* **poarta de ieșire** care ia noul vector de stare și decide care dintre componentele sale vor fi utilizate pentru a produce noul vector ascuns $h_t$.

Componentele stării $c$ pot fi privite ca niște semnale care pot fi comutate pornit și oprit. De exemplu, când întâlnim numele *Alice* în secvență, presupunem că se referă la o femeie și ridicăm semnalul din stare care spune că avem un substantiv feminin în propoziție. Când întâlnim mai departe cuvintele *și Tom*, vom ridica semnalul care spune că avem un substantiv la plural. Astfel, prin manipularea stării putem urmări proprietățile gramaticale ale propoziției.

> **Notă**: Iată o resursă excelentă pentru a înțelege detaliile interne ale LSTM-urilor: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) de Christopher Olah.

În timp ce structura internă a unei celule LSTM poate părea complexă, Keras ascunde această implementare în interiorul stratului `LSTM`, așa că singurul lucru pe care trebuie să îl facem în exemplul de mai sus este să înlocuim stratul recurent:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Notă** că antrenarea LSTM-urilor este, de asemenea, destul de lentă și este posibil să nu observați o creștere semnificativă a acurateței la începutul antrenamentului. Este posibil să fie necesar să continuați antrenamentul pentru o perioadă de timp pentru a obține o acuratețe bună.


## RNN-uri bidirecționale și multilayer

În exemplele noastre de până acum, rețelele recurente operează de la începutul unei secvențe până la sfârșit. Acest lucru ni se pare natural deoarece urmează aceeași direcție în care citim sau ascultăm vorbirea. Totuși, pentru scenarii care necesită acces aleatoriu la secvența de intrare, are mai mult sens să rulăm calculele recurente în ambele direcții. RNN-urile care permit calcule în ambele direcții se numesc RNN-uri **bidirecționale**, și pot fi create învelișând stratul recurent cu un strat special `Bidirectonal`.

> **Notă**: Stratului `Bidirectional` îi face două copii ale stratului dinăuntru, și setează proprietatea `go_backwards` a uneia dintre acele copii la `True`, făcând-o să meargă în direcția opusă de-a lungul secvenței.

Rețelele recurente, unidirecționale sau bidirecționale, captează tipare în cadrul unei secvențe și le stochează în vectori de stare sau le returnează ca ieșire. Ca și în cazul rețelelor convoluționale, putem construi un alt strat recurent după primul pentru a capta tipare de nivel mai înalt, construite din tipare de nivel mai scăzut extrase de primul strat. Aceasta ne conduce la noțiunea de **RNN multilayer**, care constă din două sau mai multe rețele recurente, unde ieșirea stratului anterior este transmisă ca intrare stratului următor.

![Imaginea arată un RNN multilayer long-short-term-memory](../../../../../translated_images/ro/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Imagine din [această postare minunată](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) de Fernando López.*

Keras face construirea acestor rețele o sarcină ușoară, pentru că trebuie doar să adăugăm mai multe straturi recurente la model. Pentru toate straturile, cu excepția ultimului, trebuie să specificăm parametrul `return_sequences=True`, pentru că avem nevoie ca stratul să returneze toate stările intermediare, și nu doar starea finală a calculului recurent.

Să construim un LSTM bidirecțional cu două straturi pentru problema noastră de clasificare.

> **Notă** acest cod din nou durează destul de mult să fie executat, dar ne oferă cea mai mare acuratețe pe care am văzut-o până acum. Deci poate merită să așteptăm și să vedem rezultatul.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN-uri pentru alte sarcini

Până acum, ne-am concentrat pe utilizarea RNN-urilor pentru a clasifica secvențe de text. Dar ele pot gestiona multe alte sarcini, cum ar fi generarea de text și traducerea automată &mdash; vom lua în considerare aceste sarcini în unitatea următoare.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
